# Przepraszam, z kim rozmawiam?
Wariant **B**  
Projekt wykonuje: **Łukasz Walancik**

In [2]:
import io
import zipfile
import urllib.request
from pathlib import Path
import pandas as pd

# --- Params ---
participant_id = 'MW_SSH_05'
transcripts_dir = Path('transcripts')  # extracted CSVs will be placed here
transcripts_dir.mkdir(parents=True, exist_ok=True)

# --- Ensure TRANSCRIPTS.zip is downloaded & extracted ---
zip_url = 'https://zenodo.org/records/15222484/files/TRANSCRIPTS.zip?download=1'
expected_csv = transcripts_dir / f'{participant_id}.csv'

if not expected_csv.exists():
    print('Downloading TRANSCRIPTS.zip from Zenodo...')
    with urllib.request.urlopen(zip_url) as resp:
        data = resp.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        zf.extractall(transcripts_dir)
    print(f'Extracted transcripts to: {transcripts_dir.resolve()}')
else:
    print('Transcripts already present, skipping download.')

# --- Load problems and responses from Zenodo ---
problems = pd.read_csv('https://zenodo.org/records/15222484/files/PROBLEMS.csv')
responses = pd.read_csv('https://zenodo.org/records/15222484/files/PROBLEMS_RESPONSES.csv')
merged_problems = pd.merge(responses, problems, on='problem_id', how='inner')

# Use column names as defined in the dataset docs
prediction_features = ['participant_decision', 'participant_certaintity', 'model_class', 'model_probability']
print(merged_problems[merged_problems['participant_id'] == participant_id][prediction_features])

# --- Retrieve transcript text for problem __P1__ for the participant ---
transcripts = pd.read_csv(expected_csv)
# Forward-fill problem markers within the participant's transcript
if 'problem_id' in transcripts.columns:
    transcripts['problem_id'] = transcripts['problem_id'].ffill()
    subset = transcripts[transcripts['problem_id'] == '__P1__']
else:
    subset = pd.DataFrame()

# Show key columns if present
cols = [c for c in ['speaker_id', 'slide_id', 'question_id', 'problem_id', 'text'] if c in subset.columns]
print(subset[cols] if cols else subset)

Transcripts already present, skipping download.
   participant_decision participant_certaintity model_class model_probability
57              trujący           średnio pewny     jadalny              0,54
58              jadalny           średnio pewny     jadalny              0,95
59              jadalny      zdecydowanie pewny     trujący                 1
    speaker_id slide_id question_id problem_id  \
180  MW_SSH_05      NaN         NaN     __P1__   
181         MW      NaN         NaN     __P1__   
182  MW_SSH_05      NaN         NaN     __P1__   

                                                  text  
180        Ok. Średnica 17,30... Mogę po całej kartce?  
181               Jasne, proszę sobie mazać spokojnie.  
182  Ok to wróćmy do… To była wysokość, średnica ka...  


## Zapoznanie się z danymi

Myślę, że warto zrobić podsumowanie tego jak dostępne dane faktycznie wygladają. Z opisu wynikałoby, że każdy użytkownik odpowiedział na 17 pytań, ale czy tak jest na pewno? Albo, że każdy, poza DE rozwiązał 3 problemy, ale czy na pewno? Ciekawią mnie też metadane, jak długość wszystkich wypowiedzi osoby badanej w transkrypcie, średnia długość odpowiedzi na pytanie, średnia długość wypowiedzi na slajd, itd.

In [3]:
from collections import Counter

### Ile plików mamy, i dlaczego nie 39?

In [4]:
licznik = Counter()
for transcript in transcripts_dir.glob('*.csv'):
  klasa = transcript.name.split('_')[1]
  licznik[klasa] += 1
print(licznik)
print(licznik.total())

Counter({'SSH': 18, 'DE': 12, 'IT': 8})
38
